# **Mount Drive**

# **Load Data**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)



In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
    detrended_yield = detrend(yield_series)
    return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])
dfy.iloc[0:5]

In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
df_fail = df_fail[df_fail.fail_share > 0]

formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers

df_fail.year.unique()

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers
df_svi.iloc[0:5,:]

In [ ]:
fpath1 = '<DATA_ROOT>/WeatherIndex/Temp_Tercile_11nov.csv'
dft = pd.read_csv(fpath1)
dft = dft.rename(columns={'Year':'year'})
print(dft.iloc[0:2,:])

fpath2 = '<DATA_ROOT>/WeatherIndex/Drought_Map.csv'
dfd = pd.read_csv(fpath2)
dfd = dfd.rename(columns={'Year':'year', 'STATE':'State', 'COUNTY':'County'})
fpath = '<DATA_ROOT>/WeatherIndex/svi2020fips.csv'
df_fips_obj = pd.read_csv(fpath)[['FIPS',	'OBJECTID']]
dfd = pd.merge(dfd,df_fips_obj,on=['OBJECTID'])
print(dfd.iloc[0:2,:])

In [ ]:
drought_indices = ['PDSI', 'SPEI3', 'SPEI6', 'SPEI9', 'SPEI12', 'SPI3', 'SPI6', 'SPI9', 'SPI12']
for idx in drought_indices:
  dfd.loc[dfd[idx] == 0,idx] = 1
print(dfd.iloc[0:2,:])

HW_indices = ['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']
for idx in HW_indices:
  dft.loc[dft[idx] == 0,idx] = 1
print(dft.iloc[0:2,:])


In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
# formatted_numbers = [int(n) for n in counties['FIPS'].values]
# counties['FIPS'] = formatted_numbers
counties.plot()

In [ ]:
ClimReg_dir = '<DATA_ROOT>/US_ClimateRegions/US_ClimateRegions/'
gdf_clim = gpd.read_file(ClimReg_dir)
gdf_clim

In [ ]:
# import zipfile
# zip_path = '<DATA_ROOT>/CONUS_States/CONUS_STATES.zip'
# extract_dir = '<DATA_ROOT>/CONUS_States/'
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

In [ ]:
states_dir = '<DATA_ROOT>/CONUS_States/'
gdf_states = gpd.read_file(states_dir)
gdf_states

# **Map Plots**

In [ ]:
cond_irig = df_fail['Irrigation Practice'] == 'ALL'
cond_fail = df_fail['fail_share'] > 0
df_fail = df_fail[cond_irig & cond_fail]
df_cnt_fail = df_fail.groupby(['FIPS','year']).first().reset_index()
cnt_frq_fail = df_cnt_fail.groupby('FIPS').count().reset_index()
cnt_frq_fail = pd.merge(counties.drop(columns=['State','County']),cnt_frq_fail,on='FIPS')
print(cnt_frq_fail.fail_share.max() , cnt_frq_fail.fail_share.min())


15 1


In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
# colors = ['#fff7ec','#fee8c8','#fdd49e','#fdbb84','#fc8d59','#ef6548','#d7301f','#990000']
colors = ['#ffffcc','#ffeda0','#fed976','#feb24c','#fd8d3c','#fc4e2a','#e31a1c','#b10026']

cmap = ListedColormap(colors)
range_values = cnt_frq_fail.quantile(list(np.arange(0,9) * 12.5 / 100))
range_values = list(range_values.fail_share)
norm = BoundaryNorm(range_values, len(colors))

fig , ax = plt.subplots(1,1,dpi=1200)
cnt_frq_fail.plot(column='fail_share',ax=ax,cmap=cmap, norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)
ax.set_xticks([])
ax.set_yticks([])

# out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
# plt.savefig(out_path + 'Map_Counties_Crop_Failure_Number' + '.png' , dpi=1200)
# plt.close()


# Rose Chart - old

In [ ]:
counties_cntr = counties.copy()
counties_cntr['geometry'] = counties_cntr['geometry'].centroid
joined_gdf = gpd.sjoin(counties_cntr, gdf_clim, how='left')
gdf_counties_clim = joined_gdf[['State', 'County', 'FIPS',	'ClimRegNam', 'ClimRegCod','geometry']]
gdf_counties_clim.iloc[0:5]

In [ ]:
df_merge = pd.merge(gdf_counties_clim.drop(columns=['State','County']),df_fail,on='FIPS')
cond_irig = df_merge['Irrigation Practice'] == 'ALL'

clim_reg_crop_fail_group = df_merge[cond_irig].groupby(['ClimRegCod','Crop'])
clim_reg_crop_fail_agg = clim_reg_crop_fail_group.agg({'fail_share': ['mean', 'count']}).reset_index()

clim_reg_crop_fail_amp = clim_reg_crop_fail_agg.pivot(
    index='ClimRegCod',columns='Crop',values=('fail_share',  'mean'))
clim_reg_crop_fail_frq = clim_reg_crop_fail_agg.pivot(
    index='ClimRegCod',columns='Crop',values=('fail_share',  'count'))

clim_reg_crop_fail_frq
# cnt_frq_fail = cnt_frq_fail.fail_share.reset_index()
# cnt_frq_fail = pd.merge(counties.drop(columns=['State','County']),cnt_frq_fail,on='FIPS')


In [ ]:
clim_reg_crop_fail_amp = clim_reg_crop_fail_amp.fillna(0)
# frq : Angle , amp : radius

for creg in list(clim_reg_crop_fail_amp.index):
  creg = 102.0
  creg_amp = clim_reg_crop_fail_amp.loc[creg,:]
  creg_frq = clim_reg_crop_fail_frq.loc[creg,:]
  df_creg = pd.merge(
      creg_amp.transpose().reset_index().rename(columns={creg:'amp'}),
      creg_frq.transpose().reset_index().rename(columns={creg:'frq'}),on='Crop')
  df_creg = df_creg.round(4)
  df_creg = df_creg.fillna(df_creg.min())
  df_creg['frq'] = df_creg['frq'] * 360 / df_creg['frq'].sum()
  # df_creg = df_creg.round(4)
  df_creg['Angle0'] = df_creg['frq'].cumsum().shift(1).fillna(0)

  fig, ax = plt.subplots(subplot_kw=dict(polar=True))
  bars = ax.bar(
      np.radians(np.array(df_creg['Angle0'])), df_creg['amp'],
      width = np.radians(np.array(df_creg['frq'])), align='center')

  # Add labels
  ax.set_theta_zero_location('N')
  ax.set_theta_direction(-1)
  ax.set_xticks(np.radians(np.array(df_creg['Angle0'])))
  ax.set_xticklabels(df_creg['Crop'])
  ax.set_rlabel_position(90)

  plt.show()

  break


In [ ]:
df_creg

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Create a sample DataFrame
data = {'Direction': ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'],
        'Value': [20, 20, 20, 20, 20, 20, 20, 20],
        'Angle': [45, 50, 40, 45, 50, 40, 35, 55]}
df = pd.DataFrame(data)
df['Angle0'] = df['Angle'].cumsum().shift(1).fillna(0)

# Convert the direction to radians for plotting
directions = np.arange(len(df))
values = df['Value']

# Create a rose chart
fig, ax = plt.subplots(subplot_kw=dict(polar=True))
# bars = ax.bar(np.radians(directions * 45), values, width=np.radians(45), align='center')
bars = ax.bar(np.radians(np.array(df['Angle0'])), values, width=np.radians(np.array(df['Angle'])), align='center')

# Add labels
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
# ax.set_xticks(np.radians(directions * 45))
ax.set_xticks(np.radians(np.array(df['Angle0'])))
ax.set_xticklabels(df['Direction'])
ax.set_rlabel_position(90)

# Add legend
# ax.legend(bars, df['Direction'], loc='upper right')

# Show the plot
plt.show()


In [ ]:
np.linspace(0, 2 * np.pi, len(df), endpoint=False)

array([0.        , 0.78539816, 1.65806279, 2.35619449, 3.14159265,
       4.01425728, 4.71238898, 5.32325422])

In [ ]:
np.radians(np.array(df['Angle']))

array([0.78539816, 0.87266463, 0.6981317 , 0.78539816, 0.87266463,
       0.6981317 , 0.61086524, 0.95993109])

In [ ]:
1.65806279 + 0.6981317

2.35619449

# **Stacked Bar Chart**

In [ ]:
farea = '<DATA_ROOT>/WeatherIndex/AreaPercentExposedHW_M_2008.csv'
df_area_HW = pd.read_csv(farea)
df_area_HW.Date = pd.to_datetime(df_area_HW.Date)
df_area_HW = df_area_HW.set_index('Date')
df_area_HW = df_area_HW.rolling(3,1,True).mean()
# df_area_HW = df_area_HW.resample('3MS').mean()
df_area_HW

,Area_Percent
Date,
2008-01-01,2.190000
2008-02-01,1.460000
2008-03-01,3.596667
2008-04-01,3.563333
2008-05-01,11.046667
...,...
2023-06-01,29.353333
2023-07-01,33.083333
2023-08-01,42.103333


In [ ]:
percent_area = [3.1,10.8,11.3,20.1,83.7,7.9,35.9,34.3,48.5,36.8,23.8,25.3,34.9,34.7,34.5,35.6]
years = np.arange(2008,2024)

df_area_HW = pd.DataFrame(columns=['Date','Area_Percent'])
date_range = pd.date_range(start='2008-01-01', end='2023-01-01', freq='YS') + pd.DateOffset(months=6)
df_area_HW.Date = list(date_range)
df_area_HW.Area_Percent = percent_area
df_area_HW = df_area_HW.set_index('Date')
# df_area_HW = df_area_HW.rolling(window=3,min_periods=1).mean()
df_area_HW.plot()

In [ ]:
file_usdm = '<DATA_ROOT>/USDM-Total.csv'
df_USDM = pd.read_csv(file_usdm)
df_USDM['ValidStart'] = pd.to_datetime(df_USDM['ValidStart'])
# df_USDM['ValidEnd'] = pd.to_datetime(df_USDM['ValidEnd'])
df_USDM = df_USDM.set_index('ValidStart')[['D0' ,	'D1' ,	'D2' ,	'D3' ,	'D4']]
df_USDM = df_USDM[df_USDM.index.year > 2007]

In [ ]:
df_fail_year_total = df_fail.groupby(['Crop','year']).count().reset_index()[['Crop','year','fail_share']]
df_fail_year_total = df_fail_year_total.pivot(index='year',columns='Crop',values='fail_share')
# df_fail_year_total.loc[2008,:] = 0
# df_fail_year_total.loc[2024,:] = 0
crops_ordered = ['WHEAT', 'CORN' , 'SOYBEANS', 'COTTON' , 'SORGHUM', 'HAY' , 'OATS', 'BARLEY' ]
df_fail_year_total = df_fail_year_total[crops_ordered]
df_fail_year_total = df_fail_year_total.sort_index()
date_range = pd.date_range(start='2009-01-01', end='2023-01-01', freq='YS') + pd.DateOffset(months=6)
# date_range = pd.date_range(start='2008-01-01', end='2024-01-01', freq='YS')
df_fail_year_total = df_fail_year_total.set_index(date_range)

colors_hex = {
    'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
    'CORN': '#008000',        # Dark Green for corn
    'SOYBEANS': '#00ff00',    # Bright Green for soybeans
    'COTTON': '#bdbd37',      # Light Yellow for cotton
    'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
    'HAY': '#785705',         # Dark Goldenrod for hay
    'OATS': '#ccc9c4',        # Bisque for oats
    'BARLEY': '#f56c1d'       # Sandy Brown for barley
}

fig , ax = plt.subplots(1,1,figsize=(18, 8),dpi=1200)
colors = ['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026']
ax.fill_between(df_USDM.index,df_USDM.D0,color=colors[0])
ax.fill_between(df_USDM.index,df_USDM.D1,color=colors[1])
ax.fill_between(df_USDM.index,df_USDM.D2,color=colors[2])
ax.fill_between(df_USDM.index,df_USDM.D3,color=colors[3])
ax.fill_between(df_USDM.index,df_USDM.D4,color=colors[4])
ax.set_ylim(0,90)
ax.tick_params(axis='both', labelsize=12)
ax.set_ylabel('Total Percent Land Area')

ax.plot(df_area_HW.index, df_area_HW.Area_Percent,color='black',linewidth=3)


ax = ax.twinx()
bottom = np.zeros(len(df_fail_year_total.index))
for col in df_fail_year_total.columns:
    ax.bar(df_fail_year_total.index, df_fail_year_total[col], width = 100, bottom=bottom,color=colors_hex.get(col))
    bottom += np.array(df_fail_year_total[col])
ax.set_ylim(0,5000)
ax.tick_params(axis='both', labelsize=12)
ax.set_ylabel('Number of Failure Events')
ax.set_xlabel('Year')

out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
plt.savefig(out_path + 'StackedBarChart_AreaChart_AllCropsFailure_DroughtArea_HWArea' + '.png' , dpi=1200)
plt.close()

# **SVI RPL_THEMES Map Plot**

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

df_svi_mean = df_svi.groupby(['FIPS'])['RPL_THEMES'].mean().reset_index()
cnt_svi = pd.merge(counties,df_svi_mean,on=['FIPS'])
cnt_svi = cnt_svi.drop(columns=['State',	'County'])

cnt_svi_theme = cnt_svi[['FIPS','RPL_THEMES']].set_index('FIPS')

bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
cnt_svi_theme = cnt_svi_theme.apply(lambda x: pd.cut(x, bins=bins, labels=labels),axis=0)
cnt_svi = cnt_svi.set_index('FIPS')
cnt_svi['RPL_THEMES'] = cnt_svi_theme['RPL_THEMES']

colors =['#fff6ec','#fbd49b','#ed654d','#820f0a']
cmap = ListedColormap(colors)
fig , ax = plt.subplots(1,1,dpi=1200)
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
cnt_svi.plot(column='RPL_THEMES',cmap=cmap,linewidth=0.1,ax=ax,edgecolor='black',
             legend=True, legend_kwds={'loc':'lower right'})
gdf_states.plot(ax=ax, color='none', edgecolor='black', linewidth=0.4)

ax.set_xticks([])
ax.set_yticks([])

# out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
# plt.savefig(out_path + 'Map_Counties_SVI_RPL_THEMES' + '.png' , dpi=1200)
# plt.close()

In [ ]:
cnt_svi

# **SVI RPL_THEMES Trend Map Plot**

In [ ]:
from scipy.stats import linregress

# cond = df_svi['FIPS'] == '01001'
# df_svi[cond][['year','RPL_THEMES']].sort_values('year').set_index('year').plot()

def cal_trend(group):
    group = group.sort_values('year')
    slope, intercept, r_value, p_value, std_err = linregress(group['year'], group['RPL_THEMES'])
    return slope

df_svi_group = df_svi[['FIPS','year','RPL_THEMES']].groupby(['FIPS'])
df_svi_slope = df_svi_group.apply(cal_trend)
df_svi_slope = df_svi_slope.reset_index().rename(columns={0:'slope'})
df_svi_slope

cnt_svi_slope = pd.merge(counties,df_svi_slope,on=['FIPS'])

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
colors = ['#8c510a','#bf812d','#dfc27d','#f6e8c3','#f5f5f5','#f5f5f5','#c7eae5','#80cdc1','#35978f','#01665e']
colors = colors[::-1]
cmap = ListedColormap(colors)
range_values = list(np.arange(-5,6)/100)
norm = BoundaryNorm(range_values, len(colors))

fig , ax = plt.subplots(1,1,dpi=1200)
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
cnt_svi_slope.plot(column='slope',cmap=cmap, norm=norm,linewidth=0.1,ax=ax,edgecolor='black')
gdf_states.plot(ax=ax, color='none', edgecolor='black', linewidth=0.4)

ax.set_xticks([])
ax.set_yticks([])

cax = fig.add_axes([0.8, 0.3, 0.02, 0.25])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
cbar.ax.tick_params(labelsize=6)

# out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
# plt.savefig(out_path + 'Map_Counties_SVI_RPL_THEMES_Trend' + '.png' , dpi=1200)

# **Yield Decrease Plot**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

In [ ]:
states_dir = '<DATA_ROOT>/CONUS_States/'
gdf_states = gpd.read_file(states_dir)
# gdf_states

In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
# formatted_numbers = [int(n) for n in counties['FIPS'].values]
# counties['FIPS'] = formatted_numbers

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
    detrended_yield = detrend(yield_series)
    return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])
dfy = dfy[dfy.year>2008]


In [ ]:
cond_irig = dfy['prodn_practice_desc'] == 'ALL PRODUCTION PRACTICES'
dfy = dfy[cond_irig]
dfy['FIPS'] = dfy['FIPS'].astype(str)
dfy = dfy[['FIPS', 'commodity_desc', 'year', 'detrended_yield']]
dfy = dfy.sort_values(['FIPS', 'commodity_desc', 'year'])
dfy

In [ ]:
def cal_change(group):
    return group.diff()

dfy['diff_yield'] = dfy.groupby(['FIPS', 'commodity_desc'])['detrended_yield'].transform(cal_change)
dfy = dfy.dropna()
dfy = dfy[dfy.diff_yield<0]
dfy

In [ ]:

df_ylddiff_count = dfy.groupby(['FIPS','year']).first().reset_index()
df_ylddiff_count = df_ylddiff_count.groupby('FIPS').count().reset_index()
cnt_ylddiff_count = pd.merge(counties,df_ylddiff_count,on=['FIPS'])
print(cnt_ylddiff_count.diff_yield.max() , cnt_ylddiff_count.diff_yield.min())

13 1


In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
# colors = ['#8c510a','#bf812d','#dfc27d','#f6e8c3','#f5f5f5','#f5f5f5','#c7eae5','#80cdc1','#35978f','#01665e']
# colors = ['#ffffb2','#fed976','#feb24c','#fd8d3c','#fc4e2a','#e31a1c','#b10026']
colors = ['#ffffcc','#ffeda0','#fed976','#feb24c','#fd8d3c','#fc4e2a','#e31a1c','#b10026']

colors = colors
cmap = ListedColormap(colors)
range_values = cnt_ylddiff_count.quantile(list(np.arange(0,9) * 12.5 / 100))
range_values = list(range_values.diff_yield)
norm = BoundaryNorm(range_values, len(colors))
cnt_ylddiff_count
fig , ax = plt.subplots(1,1,dpi=1200)

cnt_ylddiff_count.plot(column='diff_yield',ax=ax,cmap=cmap, norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)

ax.set_xticks([])
ax.set_yticks([])

out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
plt.savefig(out_path + 'Map_Counties_Yield_Decrease_count' + '.png' , dpi=1200)
plt.close()


In [ ]:
diff_yield = pd.read_csv('/content/diff_yield.csv')
failshare = pd.read_csv('/content/failshare.csv')

df = pd.merge(failshare,diff_yield,on=['FIPS'])[['FIPS','fail_share','diff_yield']]#.set_index('FIPS')
# df = df[df.fail_share>1]
# df = df[df.diff_yield>1]

correlation = df['fail_share'].corr(df['diff_yield'])
print(correlation)

df = df.groupby(['fail_share','diff_yield']).count().reset_index()
df['FIPS'] = df['FIPS'] * 4
# df = df[df.fail_share>4]

# df['fail_share'].plot()
# df['diff_yield'].plot()
df.plot(x='fail_share', y='diff_yield', kind='scatter' , s='FIPS')

In [ ]:
df

# **Map Plot HeatWave & Drought**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)



In [ ]:
# !wget https://hazards.fema.gov/nri/Content/StaticDocuments/DataDownload/Archive/v118_1/NRI_Shapefile_Counties.zip
# !mv /content/NRI_Shapefile_Counties.zip <DATA_ROOT>/WeatherIndex/NRI_Shapefile_Counties/

In [ ]:
# import zipfile
# zip_file_path = '<DATA_ROOT>/WeatherIndex/NRI_Shapefile_Counties/NRI_Shapefile_Counties.zip'
# extracted_dir = '<DATA_ROOT>/WeatherIndex/NRI_Shapefile_Counties/'
# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(extracted_dir)


In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
# formatted_numbers = [int(n) for n in counties['FIPS'].values]
# counties['FIPS'] = formatted_numbers
print(counties.geometry.unary_union.bounds)
counties.plot()

In [ ]:
shp_path = '<DATA_ROOT>/WeatherIndex/NRI_Shapefile_Counties/'
nri_counties = gpd.read_file(shp_path)
nri_counties.iloc[0:5,:]

In [ ]:
list(nri_counties.columns)

In [ ]:
target_crs = 'EPSG:4326'
nri_counties = nri_counties.to_crs(target_crs)

minx, miny, maxx, maxy = counties.geometry.unary_union.bounds
nri_counties = nri_counties.cx[minx:maxx, miny:maxy]


In [ ]:
labels =
nri_counties.plot(column='HWAV_AFREQ',cmap='Reds',legend=True)#,vmin=0,vmax=222)

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
# colors = ['#fbece4','#f5c5af','#ef977f','#da6f65','#b15a57']
colors = ['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026']
bins = [0,0.54,1.77,3.7,6.5,15.5]

cmap = ListedColormap(colors)
range_values = bins
norm = BoundaryNorm(range_values, len(colors))

fig , ax = plt.subplots(1,1,dpi=1200)
nri_counties.plot(column='HWAV_AFREQ',ax=ax,cmap=cmap, norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)
ax.set_xticks([])
ax.set_yticks([])

# out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
# plt.savefig(out_path + 'Map_Counties_HeatWave_AnnualFreq' + '.png' , dpi=1200)
# plt.close()

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
# colors = ['#fbece4','#f5c5af','#ef977f','#da6f65','#b15a57']
colors = ['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026']
bins = [0,10.7,27.5,49,74,128]

cmap = ListedColormap(colors)
range_values = bins
norm = BoundaryNorm(range_values, len(colors))

fig , ax = plt.subplots(1,1,dpi=1200)
nri_counties.plot(column='DRGT_AFREQ',ax=ax,cmap=cmap, norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)
ax.set_xticks([])
ax.set_yticks([])

# out_path = '<DATA_ROOT>/Final_Exports/20231119-Section1/'
# plt.savefig(out_path + 'Map_Counties_Drought_AnnualFreq' + '.png' , dpi=1200)
# plt.close()